In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('../python_code').resolve()))
import numpy as np
import cv2
import csv
from scipy.stats import rankdata
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline
from DataGenerator import HyperspectralTorchDataset
from BandAttentionModel import BandAttentionModel

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_path_infected = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Infected/Tray_2_row_0_column_12.npy")
image_path_healthy = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Healthy/Tray_2_row_17_column_8.npy")
model_path = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Models/BandAttentionModel_1-64-127-190-253-316-379-442-505-568-631-bands-0.985-accuracy.pt")
# model_path = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Models/BandAttentionModel_1-64-127-190-253-316-379-442-505-568-631-bands-0.957-accuracy.pt")
seeds_root = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds")

# --- Infer Bands and Label ---
model_name = model_path.stem
bands_str = model_name.split("_")[1].split("-bands")[0]
bands = list(map(int, bands_str.split("-")))
# label = 0 if "Healthy" in str(image_path) else 1
# print(bands, label)

# --- Prepare Image ---
image_infected = np.load(image_path_infected)
height, width = image_infected.shape[:2]
shape = (height, width, len(bands))

# dataset = HyperspectralTorchDataset([str(image_path)], [label], bands, shape)
dataset = HyperspectralTorchDataset([str(image_path_infected), str(image_path_healthy)], [1,0], bands, shape)
loader = DataLoader(dataset, batch_size=2, shuffle=False)

for image, _ in loader:
    image = image.to(device)

In [193]:
attention_model = BandAttentionModel(len(bands))

# Load the state dict
state = torch.load(model_path, map_location=device)

# Strip the prefix, if needed (for example, '_orig_mod.')
state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}

# Get current model state_dict
model_state = attention_model.state_dict()

# Update only the matching keys
for key in model_state.keys():
    if key in state and state[key].shape == model_state[key].shape:
        model_state[key] = state[key]  # Copy the matching weights

# Load the state_dict into the model
attention_model.load_state_dict(model_state)

# Now the model is partially loaded with compatible weights
# model = torch.compile(model, backend="eager")
attention_model.to(device).eval()

logits, attn_weights = attention_model(image)

C:\Users\sagig\AppData\Local\Temp\ipykernel_12928\944147639.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(model_path, map_location=device)


In [195]:
avg_attn = attn_weights.mean(dim=0).detach().numpy()
print(avg_attn.shape)

(11, 11)


In [196]:
import itertools
import numpy as np

def top_and_bottom_n_combinations(avg_attn: np.ndarray, band_number: int = 3, combination_number: int = 5):
    C = avg_attn.shape[0]
    scores = []

    # Calculate score for each band combination
    for combo in itertools.combinations(range(C), band_number):
        submatrix = avg_attn[np.ix_(combo, combo)]
        score = submatrix.sum()
        scores.append((combo, score))
    
    print("Last submatrix example:")
    print(submatrix)

    # Sort by score
    scores.sort(key=lambda x: x[1], reverse=True)

    # Take top and bottom k
    top_combos = scores[:combination_number]
    bottom_combos = scores[-combination_number:]

    return top_combos, bottom_combos


# Usage example:
top_combos, bottom_combos = top_and_bottom_n_combinations(avg_attn, band_number=3, combination_number=20)

print("Best combination:")
combo, score = top_combos[0]
print("Bands:", combo, "Score:", score)

print("Worst combination:")
combo, score = bottom_combos[-1]
print("Bands:", combo, "Score:", score)

Last submatrix example:
[[0.06213746 0.07296803 0.12429313]
 [0.0622757  0.07355651 0.12721446]
 [0.06250924 0.07403453 0.12894061]]
Best combination:
Bands: (0, 1, 10) Score: 1.6792737
Worst combination:
Bands: (3, 5, 7) Score: 0.26809916


In [197]:
import pandas as pd

def avg_accuracy_from_csv(csv_path, top_combos, bottom_combos, band_numbers):
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Make sure band column is string
    df["bands"] = df["bands"].astype(str)
    
    # Convert combos to strings
    top_band_strs = ["-".join(str(band_numbers[i]) for i in c) for c, _ in top_combos]
    bottom_band_strs = ["-".join(str(band_numbers[i]) for i in c) for c, _ in bottom_combos]
    
    # Get accuracies
    top_accs = df[df["bands"].isin(top_band_strs)]["accuracy"].tolist()
    bottom_accs = df[df["bands"].isin(bottom_band_strs)]["accuracy"].tolist()
    
    # Compute averages
    top_avg = np.mean(top_accs) if top_accs else None
    bottom_avg = np.mean(bottom_accs) if bottom_accs else None
    
    return top_avg, bottom_avg

band_numbers = range(1, 639, 63)
top_avg, bottom_avg = avg_accuracy_from_csv("../Models/HyperspectralMultiCNN_bands[1-64-127-190-253-316-379-442-505-568-631]_combo3_total165.csv", top_combos, bottom_combos, band_numbers)
print("Average accuracy of top combinations:", top_avg)
print("Average accuracy of bottom combinations:", bottom_avg)

Average accuracy of top combinations: 0.933215
Average accuracy of bottom combinations: 0.9412800000000001
